In [50]:
# imports for minimodel
import os
import copy
import numpy as np
import torch
import argparse
from minimodel import data
from minimodel import model_builder
from minimodel import model_trainer
from minimodel import model_trainer_exp
from minimodel import metrics

In [51]:
os.getcwd()

'/mnt/vast-nhr/projects/bthesis_cidas_richter/benjamin/minimodel/internship'

In [52]:
import torch
import torchvision
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())


torch: 2.5.1 cuda: 12.4
torchvision: 0.20.1
cuda available: True


In [53]:
# additional imports for experanto
from pathlib import Path
import matplotlib.pyplot as plt
from os import path

from tqdm import tqdm
from omegaconf import OmegaConf, open_dict

from experanto.datasets import ChunkDataset
from experanto.dataloaders import get_multisession_dataloader

In [54]:
cfg = {
    "dataset": {
        "global_sampling_rate": None,
        "global_chunk_size": None,
        "add_behavior_as_channels": False,
        "replace_nans_with_means": False,
        "cache_data": False,
        "out_keys": [
            "screen",
            "responses",
            "timestamps",
        ],
        "normalize_timestamps": True,
        "modality_config": {
            "screen": {
                "keep_nans": False,
                "sampling_rate": 30,
                "chunk_size": 1,                        # ich habe glaub ich keinen zeitlichen zusammenhang zwischen den samples
                "valid_condition": {
                    "tier": "train",
                },
                "offset": 0,
                "sample_stride": 1,
                "include_blanks": False,
                "transforms": {
                    "normalization": "normalize",
                    "Resize": {
                        "_target_": "torchvision.transforms.v2.Resize",
                        "size": [66, 130],
                    },
                },
                "interpolation": {
                    "rescale": True,
                    "rescale_size": [66, 130],
                },
            },
            "responses": {
                "keep_nans": False,
                "sampling_rate": 8,
                "chunk_size": 1,                
                "offset": 0.0,
                "transforms": {
                    "normalization": "standardize",
                },
                "interpolation": {
                    "interpolation_mode": "nearest_neighbor",
                },
                "filters": {
                    "nan_filter": {
                        "__target__": "experanto.filters.common_filters.nan_filter",
                        "__partial__": True,
                        "vicinity": 0.05,
                    },
                },
            },
        },
    },
    "dataloader": {
        "batch_size": 16,
        "shuffle": True,
        "num_workers": 4,
        "pin_memory": True,
        "drop_last": True,
        "prefetch_factor": 2,
    },
}


In [55]:
cfg_t = copy.deepcopy(cfg)
cfg_train = OmegaConf.create(cfg)
cfg_val = OmegaConf.create(cfg_t)

In [58]:
cfg_train.dataset.modality_config.screen.valid_condition = {"tier": "train"}
cfg_val.dataset.modality_config.screen.valid_condition = {"tier": "validation"}

In [59]:
from experanto.dataloaders import get_multisession_dataloader

paths = ["/mnt/vast-nhr/projects/bthesis_cidas_richter/benjamin/minimodel/internship/data_experanto/nat30k_L1_A5_022723_experanto"]
train_dl = get_multisession_dataloader(paths, cfg_train)
val_dl = get_multisession_dataloader(paths, cfg_val)

In [60]:
dataset_name, batch = next(iter(val_dl))
print(
    f"dataset: {dataset_name}",
)
for k, v in batch.items():
    # print(f"modality: {k}, shape: {v.shape}")

    print("Modality: ", k)
    if hasattr(v, "shape"):
        print("Shape:", v.shape)
    elif isinstance(v, dict):
        print("Sub-dict keys:", list(v.keys()))
    else:
        print("Type:", type(v))
    print()

# video shape: batch, times, channels, height, width
# neuronal responses: batch, times, neurons

dataset: session_0
Modality:  responses
Shape: torch.Size([16, 1, 6636])

Modality:  screen
Shape: torch.Size([16, 66, 1, 130])

Modality:  timestamps
Sub-dict keys: ['responses', 'screen']



In [63]:
print(len(val_dl))

688


In [14]:
_ ,batch = next(iter(train_dl))
NN = batch["responses"].shape[-1]       # number of neurons
print("number of neurons: ", NN)

number of neurons:  6636


In [15]:
# setup
device = torch.device('cuda')
mouse_id = 0
data_path = '../data'
weight_path = './checkpoints_16-320_exp'
results_path = './results_16-320_exp'
os.makedirs(weight_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)


In [16]:
# Building Model

nlayers = 2
nconv1 = 16
nconv2 = 320
model, in_channels = model_builder.build_model(NN=NN, n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2)
model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels)

model_path = os.path.join(weight_path, model_name)
print('model path: ', model_path)
model = model.to(device)

core shape:  torch.Size([1, 320, 33, 65])
input shape of readout:  (320, 33, 65)
model name:  l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt
model path:  ./checkpoints_16-320_exp/l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt


In [ ]:
# Training the model
print(device)
if not os.path.exists(model_path):
    best_state_dict = model_trainer_exp.train(model, train_dl=train_dl, val_dl=val_dl, device=device)
    torch.save(best_state_dict, model_path)
    print('saved model', model_path)
model.load_state_dict(torch.load(model_path))
print('loaded model', model_path)

In [ ]:
# test model
test_pred = model_trainer.test_epoch(model, img_test)
print('test_pred: ', test_pred.shape, test_pred.min(), test_pred.max())


test_fev, test_feve = metrics.feve(spks_rep_all, test_pred)
print('FEVE (test, all): ', np.mean(test_feve))

threshold = 0.15
print(f'filtering neurons with FEV > {threshold}')
valid_idxes = np.where(test_fev > threshold)[0]
print(f'valid neurons: {len(valid_idxes)} / {len(test_fev)}')
print(f'FEVE (test, FEV>0.15): {np.mean(test_feve[test_fev > threshold])}')

In [ ]:
# ---- Saving performance scores ----
file_name = "results_" + str(mouse_id)
results_file_path = os.path.join(results_path, file_name)

print(f"Results saved at: {results_file_path}")
np.savez(results_file_path, FEV_scores=test_fev, FEVE_scores=test_feve, neurons_index=ineur)